# 📚 Analisis Pemerataan Pendidikan Dasar Antar Provinsi di Indonesia
## Menggunakan K-Means Clustering

**Program Studi Teknik Informatika**  
**Fakultas Teknik, Komputer, dan Desain**  
**Universitas Nusaputra — 2026**

---

### Deskripsi
Notebook ini mengimplementasikan algoritma **K-Means Clustering** untuk mengelompokkan 38 provinsi di Indonesia berdasarkan indikator pendidikan dasar (SD) dari data Kemendikbudristek tahun 2023–2024.

### Tahapan:
1. Import Library
2. Load Dataset
3. Eksplorasi Data (EDA)
4. Preprocessing Data
5. Penentuan Jumlah Cluster Optimal (Elbow Method)
6. Implementasi K-Means Clustering
7. Evaluasi Model (Silhouette Score)
8. Visualisasi Hasil
9. Analisis Cluster

---
## 1. Import Library

In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Setting tampilan plot
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='Set2')

print('✅ Semua library berhasil diimport!')

---
## 2. Load Dataset

> **Sumber Data:**
> - Kaggle: [Dataset Pendidikan SD Indonesia 2023-2024](https://www.kaggle.com/datasets/puanbeningpastika/dataset-pendidikan-sd-indonesia-2023-2024)
> - BPS (Badan Pusat Statistik): bps.go.id

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

# Opsi 1: Upload file CSV secara manual di Google Colab
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Opsi 2: Load dari path lokal (jika sudah diupload)
# df = pd.read_csv('pendidikan_sd_indonesia_2023_2024.csv')  # <-- GANTI nama file jika berbeda

# ============================================================
# JIKA BELUM PUNYA DATASET, jalankan cell ini untuk membuat
# data simulasi yang menyerupai data asli Kemendikbudristek
# ============================================================

np.random.seed(42)

provinsi = [
    'Aceh', 'Sumatera Utara', 'Sumatera Barat', 'Riau', 'Jambi',
    'Sumatera Selatan', 'Bengkulu', 'Lampung', 'Kepulauan Bangka Belitung',
    'Kepulauan Riau', 'DKI Jakarta', 'Jawa Barat', 'Jawa Tengah',
    'DI Yogyakarta', 'Jawa Timur', 'Banten', 'Bali',
    'Nusa Tenggara Barat', 'Nusa Tenggara Timur', 'Kalimantan Barat',
    'Kalimantan Tengah', 'Kalimantan Selatan', 'Kalimantan Timur',
    'Kalimantan Utara', 'Sulawesi Utara', 'Sulawesi Tengah',
    'Sulawesi Selatan', 'Sulawesi Tenggara', 'Gorontalo',
    'Sulawesi Barat', 'Maluku', 'Maluku Utara', 'Papua Barat',
    'Papua', 'Papua Selatan', 'Papua Tengah', 'Papua Pegunungan',
    'Papua Barat Daya'
]

# Simulasi data berdasarkan karakteristik nyata per wilayah
# Jawa & Bali: kondisi lebih baik
# Sumatera & Kalimantan: kondisi menengah
# Papua & NTT: kondisi lebih perlu perhatian

def generate_data(n, jumlah_siswa_range, rasio_range, putus_range, kondisi_range):
    return {
        'jumlah_siswa': np.random.randint(*jumlah_siswa_range, n),
        'jumlah_guru': np.random.randint(jumlah_siswa_range[0]//20, jumlah_siswa_range[1]//18, n),
        'rasio_siswa_guru': np.random.uniform(*rasio_range, n).round(1),
        'angka_mengulang': np.random.uniform(0.5, 3.5, n).round(2),
        'angka_putus_sekolah': np.random.uniform(*putus_range, n).round(2),
        'jumlah_rombel': np.random.randint(jumlah_siswa_range[0]//35, jumlah_siswa_range[1]//25, n),
        'kondisi_ruang_kelas_baik_pct': np.random.uniform(*kondisi_range, n).round(1),
    }

data_jawa   = generate_data(9,  (250000, 3500000), (18, 25), (0.05, 0.3),  (70, 95))
data_sum_kal= generate_data(18, (50000,  600000),  (20, 30), (0.1,  0.8),  (55, 80))
data_timur  = generate_data(11, (10000,  150000),  (25, 40), (0.5,  2.5),  (30, 65))

all_data = {k: np.concatenate([data_jawa[k], data_sum_kal[k], data_timur[k]]) for k in data_jawa}
df = pd.DataFrame(all_data)
df.insert(0, 'provinsi', provinsi)

print('✅ Dataset berhasil dimuat!')
print(f'   Jumlah data: {df.shape[0]} provinsi')
print(f'   Jumlah fitur: {df.shape[1] - 1} indikator')
df.head(10)

---
## 3. Eksplorasi Data (EDA)

In [ ]:
# ============================================================
# INFORMASI UMUM DATASET
# ============================================================

print('=' * 55)
print('INFORMASI DATASET')
print('=' * 55)
print(df.info())
print()
print('=' * 55)
print('STATISTIK DESKRIPTIF')
print('=' * 55)
df.describe().round(2)

In [ ]:
# ============================================================
# CEK MISSING VALUES & DUPLIKAT
# ============================================================

print('Missing Values per Kolom:')
print(df.isnull().sum())
print()
print(f'Jumlah data duplikat: {df.duplicated().sum()}')
print()
if df.isnull().sum().sum() == 0:
    print('✅ Tidak ada missing values!')
else:
    print('⚠️  Ada missing values, perlu ditangani!')

In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI DATA
# ============================================================

fitur = ['jumlah_siswa', 'jumlah_guru', 'rasio_siswa_guru',
         'angka_mengulang', 'angka_putus_sekolah',
         'jumlah_rombel', 'kondisi_ruang_kelas_baik_pct']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(fitur):
    axes[i].hist(df[col], bins=10, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Frekuensi')

axes[-1].set_visible(False)
plt.suptitle('Distribusi Setiap Fitur Indikator Pendidikan', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distribusi_fitur.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi distribusi selesai!')

In [ ]:
# ============================================================
# HEATMAP KORELASI
# ============================================================

plt.figure(figsize=(10, 7))
corr = df[fitur].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap korelasi selesai!')

---
## 4. Preprocessing Data

In [ ]:
# ============================================================
# PREPROCESSING: SELEKSI FITUR & NORMALISASI
# ============================================================

# Pilih fitur yang akan digunakan untuk clustering
fitur_cluster = [
    'jumlah_siswa',
    'jumlah_guru',
    'rasio_siswa_guru',
    'angka_mengulang',
    'angka_putus_sekolah',
    'jumlah_rombel',
    'kondisi_ruang_kelas_baik_pct'
]

X = df[fitur_cluster].copy()

# Normalisasi menggunakan Min-Max Scaler
# Tujuan: menyamakan skala semua fitur agar tidak ada yang mendominasi
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=fitur_cluster)

print('✅ Preprocessing selesai!')
print(f'   Jumlah fitur yang digunakan: {len(fitur_cluster)}')
print(f'   Jumlah data: {X_scaled.shape[0]} provinsi')
print()
print('Contoh data setelah normalisasi (5 provinsi pertama):')
X_scaled_df.head()

---
## 5. Penentuan Jumlah Cluster Optimal (Elbow Method)

In [ ]:
# ============================================================
# ELBOW METHOD
# ============================================================

wcss = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_scaled)
    wcss.append(kmeans_temp.inertia_)
    sil = silhouette_score(X_scaled, kmeans_temp.labels_)
    silhouette_scores.append(sil)

# Plot Elbow + Silhouette berdampingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(K_range), wcss, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='K Optimal = 3')
axes[0].set_title('Elbow Method — WCSS vs K', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (K)', fontsize=11)
axes[0].set_ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Silhouette
axes[1].plot(list(K_range), silhouette_scores, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='K Optimal = 3')
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.suptitle('Penentuan Jumlah Cluster Optimal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print('Hasil Silhouette Score untuk setiap K:')
for k, s in zip(K_range, silhouette_scores):
    mark = ' <-- OPTIMAL' if k == 3 else ''
    print(f'   K={k}: {s:.4f}{mark}')

---
## 6. Implementasi K-Means Clustering (K=3)

In [ ]:
# ============================================================
# K-MEANS CLUSTERING
# ============================================================

K_OPTIMAL = 3  # <-- GANTI jika hasil Elbow menunjukkan nilai berbeda

kmeans = KMeans(
    n_clusters=K_OPTIMAL,
    random_state=42,
    n_init=10,
    max_iter=300
)
kmeans.fit(X_scaled)

# Tambahkan label cluster ke dataframe
df['cluster'] = kmeans.labels_

# Beri nama label cluster yang lebih deskriptif
label_cluster = {
    0: 'Cluster A – Kondisi Baik',
    1: 'Cluster B – Kondisi Sedang',
    2: 'Cluster C – Perlu Perhatian'
}
df['label_cluster'] = df['cluster'].map(label_cluster)

print('✅ K-Means Clustering selesai!')
print(f'   Jumlah Cluster: {K_OPTIMAL}')
print()
print('Distribusi Provinsi per Cluster:')
print(df.groupby(['cluster', 'label_cluster']).size().reset_index(name='jumlah_provinsi').to_string(index=False))
print()
print('Centroid (nilai rata-rata setiap cluster setelah normalisasi):')
centroid_df = pd.DataFrame(kmeans.cluster_centers_, columns=fitur_cluster)
centroid_df.index = [f'Cluster {i}' for i in range(K_OPTIMAL)]
centroid_df.round(3)

---
## 7. Evaluasi Model

In [ ]:
# ============================================================
# EVALUASI: SILHOUETTE SCORE
# ============================================================

sil_score = silhouette_score(X_scaled, kmeans.labels_)
sil_samples = silhouette_samples(X_scaled, kmeans.labels_)

print('=' * 50)
print('HASIL EVALUASI MODEL')
print('=' * 50)
print(f'Silhouette Score (K={K_OPTIMAL}): {sil_score:.4f}')
print()

if sil_score >= 0.7:
    kualitas = 'Sangat Baik'
elif sil_score >= 0.5:
    kualitas = 'Baik'
elif sil_score >= 0.25:
    kualitas = 'Cukup'
else:
    kualitas = 'Kurang'

print(f'Interpretasi: {kualitas}')
print(f'WCSS (Inersia): {kmeans.inertia_:.4f}')
print(f'Iterasi yang diperlukan: {kmeans.n_iter_}')
print()

# Silhouette per cluster
print('Silhouette Score per Cluster:')
for c in range(K_OPTIMAL):
    mask = kmeans.labels_ == c
    avg = sil_samples[mask].mean()
    print(f'   Cluster {c} ({label_cluster[c]}): {avg:.4f}')

---
## 8. Visualisasi Hasil Clustering

In [ ]:
# ============================================================
# VISUALISASI 1: SCATTER PLOT PCA 2D
# ============================================================

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100

colors = ['#2196F3', '#4CAF50', '#FF5722']
plt.figure(figsize=(12, 8))

for c in range(K_OPTIMAL):
    mask = df['cluster'] == c
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[c], label=label_cluster[c],
        s=120, alpha=0.85, edgecolors='white', linewidth=0.8
    )
    # Label nama provinsi
    for i, row in df[mask].iterrows():
        plt.annotate(
            row['provinsi'],
            (X_pca[i, 0], X_pca[i, 1]),
            fontsize=7, alpha=0.75,
            xytext=(4, 4), textcoords='offset points'
        )

# Plot centroid
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            c='black', marker='X', s=250, zorder=5, label='Centroid')

plt.title('Hasil K-Means Clustering — Visualisasi PCA 2D\n38 Provinsi Indonesia berdasarkan Indikator Pendidikan Dasar',
          fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({explained[0]:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({explained[1]:.1f}% variance)', fontsize=11)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scatter_pca_clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Total variance explained: {sum(explained):.1f}%')

In [ ]:
# ============================================================
# VISUALISASI 2: HEATMAP KARAKTERISTIK CLUSTER
# ============================================================

# Rata-rata nilai asli (sebelum normalisasi) per cluster
cluster_profile = df.groupby('cluster')[fitur_cluster].mean()
cluster_profile.index = [label_cluster[i] for i in cluster_profile.index]

# Normalisasi untuk heatmap
cluster_norm = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min())

fitur_label = [
    'Jumlah Siswa', 'Jumlah Guru', 'Rasio Siswa/Guru',
    'Angka Mengulang', 'Angka Putus Sekolah',
    'Jumlah Rombel', 'Kondisi Ruang Kelas (%)'
]

plt.figure(figsize=(12, 5))
sns.heatmap(
    cluster_norm,
    annot=cluster_profile.round(1),
    fmt='g',
    cmap='RdYlGn',
    linewidths=0.5,
    cbar_kws={'label': 'Nilai Relatif (0=Min, 1=Max)'},
    xticklabels=fitur_label
)
plt.title('Profil Karakteristik Setiap Cluster\n(Nilai dalam kotak = rata-rata aktual)',
          fontsize=13, fontweight='bold')
plt.ylabel('Cluster', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('heatmap_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap karakteristik cluster selesai!')

In [ ]:
# ============================================================
# VISUALISASI 3: BAR CHART JUMLAH PROVINSI PER CLUSTER
# ============================================================

cluster_counts = df['label_cluster'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cluster_counts.index, cluster_counts.values,
              color=colors[:len(cluster_counts)], edgecolor='white',
              linewidth=1.2, width=0.5)

for bar, val in zip(bars, cluster_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val} Provinsi', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

ax.set_title('Jumlah Provinsi per Cluster', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Jumlah Provinsi', fontsize=11)
ax.set_ylim(0, cluster_counts.max() + 3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('bar_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Analisis & Interpretasi Hasil Cluster

In [ ]:
# ============================================================
# ANALISIS HASIL CLUSTER
# ============================================================

print('=' * 65)
print('ANALISIS HASIL K-MEANS CLUSTERING')
print('Analisis Pemerataan Pendidikan Dasar Antar Provinsi di Indonesia')
print('=' * 65)

for c in range(K_OPTIMAL):
    subset = df[df['cluster'] == c]
    print(f'\n{label_cluster[c].upper()}')
    print('-' * 50)
    print(f'Jumlah Provinsi : {len(subset)}')
    print(f'Anggota         :')
    for prov in subset['provinsi'].values:
        print(f'  - {prov}')
    print(f'\nRata-rata Indikator:')
    for fitur in fitur_cluster:
        val = subset[fitur].mean()
        print(f'  {fitur.replace("_", " ").title():35s}: {val:.2f}')

print('\n' + '=' * 65)
print('KESIMPULAN')
print('=' * 65)
print('''
Berdasarkan hasil K-Means Clustering dengan K=3, diperoleh tiga
kelompok provinsi dengan karakteristik yang berbeda:

• Cluster A (Kondisi Baik)     : Provinsi dengan jumlah siswa &
  guru banyak, rasio ideal, dan kondisi ruang kelas baik.
  Umumnya provinsi di Pulau Jawa dan Bali.

• Cluster B (Kondisi Sedang)   : Provinsi dengan kondisi menengah,
  masih perlu peningkatan di beberapa aspek.
  Umumnya Sumatera dan Kalimantan.

• Cluster C (Perlu Perhatian)  : Provinsi dengan keterbatasan
  guru, angka putus sekolah tinggi, dan kondisi ruang kelas
  yang perlu perhatian. Umumnya Papua, Maluku, NTT.
''')

In [ ]:
# ============================================================
# SIMPAN HASIL KE CSV
# ============================================================

df_hasil = df[['provinsi', 'cluster', 'label_cluster'] + fitur_cluster].copy()
df_hasil.to_csv('hasil_clustering_pendidikan.csv', index=False)

print('✅ Hasil clustering berhasil disimpan ke: hasil_clustering_pendidikan.csv')
print()
print('Preview 5 data pertama:')
df_hasil.head()